# Lifetime Oλ survival validation across eight datasets
Fit continuous and midpoint-centered transition models on seven datasets and evaluate the training-derived scores in the eighth. Export held-out memberships for clinical-field validation. Then estimate each dataset’s own hazard ratios using the continuous scaling and transition definition learned from the other seven datasets. Both stages use the existing direct spectral particles.

Run in order. Training fits resume separately by fold and model. The two validation designs have separate Drive output directories.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import tempfile
import pandas as pd

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
FIRST_STAGE = DATA / 'olambda_asymptotes_min1000_no_floor_v1'
ESTIMATES = FIRST_STAGE / 'patient_olambda_asymptotes.csv'
PARTICLES = FIRST_STAGE / 'patient_olambda_particles.csv'
OUTCOMES = DATA / 'clinical_outcomes.csv'
OUTPUT = DATA / 'bayesian_Olambda_survival' / 'lifetime_no_floor_adjusted_midpoint_v1'

DRAWS, TUNE, CHAINS, CORES = 1000, 1000, 2, 2
LODO_ACCEPT, INDEPENDENT_ACCEPT, RANDOM_STATE = 0.99, 0.97, 0
ADJUSTMENTS = {'Treatment':'axicel', 'Disease':'lymphoma'}

In [ ]:
repo = Path(tempfile.mkdtemp(prefix='survival-')) / 'repo'
subprocess.run(['git', 'clone', '--quiet', '--single-branch', '--branch', 'main',
                'https://github.com/adamnik/CAR-T-spectral-organization.git', str(repo)], check=True)
REVISION = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pymc==5.28.5',
                'arviz>=0.22,<1', 'h5netcdf>=1.5'], check=True)
print('Survival model revision:', REVISION)
sys.path.insert(0, str(repo))

In [ ]:
keys = ['Dataset', 'sample_id']
est = pd.read_csv(ESTIMATES).set_index(keys, verify_integrity=True)
particles = pd.read_csv(PARTICLES).set_index(keys, verify_integrity=True)
assert est.model.eq('latent_timescale_student_t3_scale_0.35_no_asymptote_floor').all()
assert len(est) == 163 and est.index.get_level_values('Dataset').nunique() == 8
assert est.index.sort_values().equals(particles.index.sort_values())
assert list(particles) == [f'particle_{i:02d}' for i in range(1,17)]
assert (particles.gt(1) & particles.lt(float('inf'))).all().all()

from survival.survival_model import load_olambda_analysis
data, raw, _, weights, _ = load_olambda_analysis(
    ESTIMATES, OUTCOMES, 1000, 16, PARTICLES, list(ADJUSTMENTS))
assert len(data)==163 and data.Dataset.nunique()==8 and data.event_observed.sum()==105
display(data.groupby('Dataset').agg(patients=('sample_id','size'),events=('event_observed','sum')))
hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in (ESTIMATES, PARTICLES, OUTCOMES)}
def tag(config):
    return hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:12]
config = dict(revision=REVISION, inputs=hashes, adjustments=ADJUSTMENTS, draws=DRAWS, tune=TUNE,
              chains=CHAINS, target_accept=LODO_ACCEPT, random_state=RANDOM_STATE)
LODO = OUTPUT/'leave_one_dataset_out'/tag(config)
LODO.mkdir(parents=True,exist_ok=True)
(LODO/'notebook_config.json').write_text(json.dumps(config,indent=2))
print('LODO results:', LODO)

## Seven-dataset training and held-out validation
The transition prior spans the training Q10–Q90 interval. The transition width is 0.10 training SD. The held-out transition score is the training global log-HR multiplied by activation minus 0.5 (with subtraction before multiplication). Membership averages activation over training crossover draws and each held-out patient’s particles. The final two survival fits evaluate the cross-fitted risk scores.

In [ ]:
command = [sys.executable,'-u','-m','survival.validate_olambda_survival_crossfit',
           ESTIMATES, PARTICLES, OUTCOMES, LODO, '--minimum-cells',1000,'--n-particles',16,
           '--midpoint-centered','--crossover-quantile-low',0.1,'--crossover-quantile-high',0.9,
           '--transition-scale-sd',0.10,'--draws',DRAWS,'--tune',TUNE,'--chains',CHAINS,
           '--cores',CORES,'--target-accept',LODO_ACCEPT,'--random-state',RANDOM_STATE,
           '--adjustment','Treatment=axicel','--adjustment','Disease=lymphoma']
with subprocess.Popen(list(map(str,command)),cwd=repo,stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
    for line in proc.stdout:
        print(line,end='',flush=True)
    if proc.wait():
        raise subprocess.CalledProcessError(proc.returncode,command)

In [ ]:
import numpy as np
training=pd.read_csv(LODO/'leave_one_dataset_out_training_results.csv')
assert len(training)==16 and not training.duplicated(['heldout_dataset','model']).any()
display(training[['heldout_dataset','model','HR_median','HR_q025','HR_q975','divergences','max_rhat']])
display(pd.read_csv(LODO/'cross_fitted_heldout_validation_results.csv'))
membership_path=LODO/'crossfit_patient_regime_probabilities.csv'
membership=pd.read_csv(membership_path)
assert pd.MultiIndex.from_frame(membership[keys]).sort_values().equals(est.index.sort_values())
assert membership.Dataset.eq(membership.heldout_dataset).all()
assert membership.posterior_mean_probability_high.between(0,1).all()
membership['posterior_mean_probability_adverse']=membership.posterior_mean_probability_high
membership.to_csv(membership_path,index=False)
print('Clinical-field membership CSV:',membership_path)
display(membership.groupby('Dataset').agg(patients=('sample_id','size'),
        high_mass=('posterior_mean_probability_high','sum'),
        low_mass=('posterior_mean_probability_high',lambda q: (1-q).sum())))

## Independent single-dataset effects with transferred predictor definitions
For each dataset, transfer the continuous center and SD and the posterior-median transition crossover and width from its seven-dataset training fold. Center the sigmoid at 0.5. Estimate the baseline and hazard coefficient in the eighth dataset, adjusting for treatment and disease where they vary. Each completed fit is checkpointed.

In [ ]:
from shared.analysis_io import safe_name

def training_definition(dataset):
    directory = LODO / 'training_fits' / f'heldout_{safe_name(dataset)}'
    train = data.loc[data.Dataset.ne(dataset)]
    outcomes = pd.read_csv(directory / 'training_outcomes.csv')
    assert dataset not in set(outcomes.Dataset)
    assert train.Dataset.nunique() == 7
    pars = {}
    for model in ('continuous', 'smooth_transition_Q10_Q90'):
        fit = directory / model
        result = pd.read_csv(fit / 'model_result.csv').iloc[0]
        manifest = json.loads((fit / 'run_manifest.json').read_text())
        assert result.patients == len(train) and result.datasets == 7
        scale = manifest['particle_scaling']
        assert np.allclose([scale['center'], scale['scale']],
                           [train.O_lambda_inf_central.mean(), train.O_lambda_inf_central.std(ddof=1)])
        if model == 'continuous':
            pars.update(center=float(scale['center']), sd=float(scale['scale']))
        else:
            assert manifest['hazard_activation_offset'] == 0.5
            pars.update(crossover=float(result.crossover_median),
                        width=float(scale['scale'] * manifest['transition_scale_sd']))
    assert np.isfinite(list(pars.values())).all() and min(pars['sd'], pars['width']) > 0
    return pars

definitions = {d: training_definition(d) for d in sorted(data.Dataset.unique())}
ind_config = dict(config, target_accept=INDEPENDENT_ACCEPT, training_fold_definitions=definitions)
OUTPUT_DIR = OUTPUT / 'independent_dataset_seven_training_datasets' / tag(ind_config)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'notebook_config.json').write_text(json.dumps(ind_config, indent=2))
audit = pd.DataFrame.from_dict(definitions, orient='index').rename_axis('heldout_dataset')
audit.to_csv(OUTPUT_DIR / 'transferred_predictor_definitions.csv')
display(audit)
TARGET_ACCEPT = INDEPENDENT_ACCEPT
SAVE_POSTERIORS = False
print('Independent results:', OUTPUT_DIR)

In [ ]:
import gc
import time
import traceback

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
from IPython.display import display
from scipy.special import expit

from survival.survival_model import (
    ADJUSTMENT_EFFECT_SD,
    GLOBAL_EFFECT_SD,
    adjustment_parameters,
    baseline_parameters,
    diagnostics,
    effect_summary,
    interval_loglikelihood,
    load_olambda_analysis,
    marginalize_particles,
    model_coordinates,
    survival_arrays,
)


def independent_adjustments(data):
    # Dummy-code only adjustment variables that vary inside this dataset.
    columns = []
    rows = []
    for column, preferred_reference in ADJUSTMENTS.items():
        observed = data[column].astype(str)
        levels = sorted(observed.unique())
        if len(levels) < 2:
            continue
        reference = preferred_reference if preferred_reference in levels else levels[0]
        for level in levels:
            if level == reference:
                continue
            columns.append(observed.eq(level).to_numpy(float))
            rows.append({
                'adjustment': f'{column}: {level} vs {reference}',
                'column': column,
                'level': level,
                'reference': reference,
            })
    matrix = np.column_stack(columns) if columns else np.empty((len(data), 0), float)
    return matrix, pd.DataFrame(rows)


def fit_independent_effect(data, predictor_particles, particle_weights,
                           predictor, predictor_kind, model, effect_unit,
                           crossover=None, effective_high=None):
    arrays = survival_arrays(data)
    adjustment_matrix, adjustment_table = independent_adjustments(data)
    coords = model_coordinates(data, arrays)
    if len(adjustment_table):
        coords['adjustment'] = adjustment_table.adjustment.tolist()

    with pm.Model(coords=coords):
        baseline_log_rate = baseline_parameters(arrays)
        effect = pm.Normal('effect', 0.0, GLOBAL_EFFECT_SD)
        clinical_adjustment = adjustment_parameters(
            adjustment_matrix, ADJUSTMENT_EFFECT_SD
        )
        hazard = pt.exp(
            baseline_log_rate[:, None, :]
            + clinical_adjustment[:, None, None]
            + effect * pt.as_tensor_variable(predictor_particles)[:, :, None]
        )
        conditional = interval_loglikelihood(hazard, arrays)
        patient_loglikelihood = marginalize_particles(conditional, particle_weights)
        pm.Deterministic('patient_loglikelihood', patient_loglikelihood, dims='observation')
        pm.Potential('survival_loglikelihood', pt.sum(patient_loglikelihood))

        started = time.time()
        idata = pm.sample(
            tune=TUNE, draws=DRAWS, chains=CHAINS, cores=CORES,
            target_accept=TARGET_ACCEPT, init='adapt_diag',
            random_seed=RANDOM_STATE, progressbar=True,
            return_inferencedata=True,
        )
        elapsed = time.time() - started

    variables = [
        'effect', 'baseline_log_rate_initial',
        'baseline_smoothness', 'baseline_innovation',
    ]
    if len(adjustment_table):
        variables.append('adjustment_effect')
    _, diagnostic_values = diagnostics(idata, variables)
    effect_draws = np.asarray(idata.posterior.effect).reshape(-1)
    result = {
        'predictor': predictor,
        'predictor_kind': predictor_kind,
        'model': model,
        'Dataset': str(data.Dataset.iloc[0]),
        **effect_summary(effect_draws),
        **diagnostic_values,
        'direction_probability': float(max(np.mean(effect_draws > 0),
                                           np.mean(effect_draws < 0))),
        'effect_unit': effect_unit,
        'crossover': crossover,
        'patients': len(data),
        'events': int(data.event_observed.sum()),
        'adjustment_coefficients': adjustment_matrix.shape[1],
        'effective_high_patients': effective_high,
        'effective_low_patients': None if effective_high is None else len(data) - effective_high,
        'wall_time_seconds': elapsed,
    }
    if SAVE_POSTERIORS:
        posterior_dir = OUTPUT_DIR / 'posteriors'
        posterior_dir.mkdir(parents=True, exist_ok=True)
        safe_predictor = ''.join(x if x.isalnum() else '_' for x in predictor).strip('_')
        idata.to_netcdf(posterior_dir / f'{safe_predictor}__{result["Dataset"]}__{model}.nc')
    del idata
    gc.collect()
    return result

In [ ]:
results=[]
for dataset in sorted(data.Dataset.unique()):
    mask=data.Dataset.eq(dataset).to_numpy()
    subset=data.loc[mask].reset_index(drop=True)
    definition = definitions[dataset]
    continuous = (raw[mask] - definition['center']) / definition['sd']
    crossover = definition['crossover']
    activation = expit((raw[mask] - crossover) / definition['width'])
    for model,x,unit,c,high in [
            ('continuous',continuous,'per seven-dataset training SD of O_lambda',None,None),
            ('smooth_transition',activation-.5,'high versus low O_lambda plateaus',crossover,
             float((activation*weights[mask]).sum()))]:
        saved=OUTPUT_DIR/f'{dataset}__{model}.csv'
        print(f'{dataset} | {model} | {len(subset)} patients | {"resume" if saved.exists() else "fitting"}',flush=True)
        if not saved.exists():
            row=fit_independent_effect(subset,x,weights[mask],'O_lambda','system',model,unit,c,high)
            tmp=saved.with_suffix('.tmp')
            pd.DataFrame([row]).to_csv(tmp,index=False)
            tmp.replace(saved)
        results.append(pd.read_csv(saved))
        result=pd.concat(results,ignore_index=True)
        result.to_csv(OUTPUT_DIR/'independent_dataset_results.csv',index=False)
assert len(result)==16 and not result.duplicated(['Dataset','model']).any()
display(result[['Dataset','model','patients','events','HR_median','HR_q025','HR_q975','divergences','max_rhat']])
print('Independent results:',OUTPUT_DIR/'independent_dataset_results.csv')